# Data Cleaning Notebook

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path
import logging

# Set data path
DATA_DIR = Path('../data')

## 1. Load Customers Data

In [13]:
cust_df = pd.read_csv(DATA_DIR / 'customers.csv')
print(f"Initial rows: {len(cust_df)}")
cust_df.head()

Initial rows: 10


,customer_id,name,email,region,signup_date
0,C001,Alice Johnson,alice@example.com,North,2023-01-15
1,C002,Bob Smith,bob@test.com,South,2023-02-20
2,C003,Charlie Brown,NaN,East,2023-03-10
3,C004,David Wilson,david@example.com,West,2023-04-05
4,C005,Eva Garcia,NaN,Central,2023-05-12


## 1.1 Remove Duplicates
Remove duplicate rows based on `customer_id`, keeping the most recent `signup_date`.

In [14]:
# First parse dates temporarily to sort correctly
cust_df['signup_date'] = pd.to_datetime(cust_df['signup_date'], errors='coerce')
cust_df = cust_df.sort_values(by=['customer_id', 'signup_date'], ascending=[True, False])

# Drop duplicates
before_count = len(cust_df)
cust_df = cust_df.drop_duplicates(subset='customer_id', keep='first')
print(f"Rows removed: {before_count - len(cust_df)}")
cust_df.head()

Rows removed: 2


,customer_id,name,email,region,signup_date
0,C001,Alice Johnson,alice@example.com,North,2023-01-15
1,C002,Bob Smith,bob@test.com,South,2023-02-20
2,C003,Charlie Brown,NaN,East,2023-03-10
3,C004,David Wilson,david@example.com,West,2023-04-05
4,C005,Eva Garcia,NaN,Central,2023-05-12


## 1.2 Standardize Emails
Standardize email addresses to lowercase and flag missing or malformed emails.

In [15]:
cust_df['email'] = cust_df['email'].astype(str).str.lower().str.strip()
cust_df.loc[cust_df['email'].isin(['nan', 'none', '']), 'email'] = ''

# Validation flag
cust_df['is_valid_email'] = cust_df['email'].apply(lambda x: isinstance(x, str) and '@' in x and '.' in x if x else False)
cust_df[['customer_id', 'email', 'is_valid_email']].head()

,customer_id,email,is_valid_email
0,C001,alice@example.com,True
1,C002,bob@test.com,True
2,C003,NaN,False
3,C004,david@example.com,True
4,C005,NaN,False


## 1.3 Parse Signup Date
Parse signup_date into YYYY-MM-DD and log warnings for unparseable values.

In [16]:
cust_df['signup_date'] = pd.to_datetime(cust_df['signup_date'], errors='coerce')
unparseable = cust_df['signup_date'].isna().sum()
if unparseable > 0:
    print(f"WARNING: Found {unparseable} unparseable signup_date values.")

cust_df['signup_date'] = cust_df['signup_date'].dt.strftime('%Y-%m-%d')
cust_df.head()

,customer_id,name,email,region,signup_date,is_valid_email
0,C001,Alice Johnson,alice@example.com,North,2023-01-15,True
1,C002,Bob Smith,bob@test.com,South,2023-02-20,True
2,C003,Charlie Brown,NaN,East,2023-03-10,False
3,C004,David Wilson,david@example.com,West,2023-04-05,True
4,C005,Eva Garcia,NaN,Central,2023-05-12,False


## 1.4 Strip Whitespace
Strip leading/trailing whitespace from `name` and `region` columns.

In [17]:
cust_df['name'] = cust_df['name'].astype(str).str.strip()
cust_df['region'] = cust_df['region'].astype(str).str.strip()
cust_df.head()

,customer_id,name,email,region,signup_date,is_valid_email
0,C001,Alice Johnson,alice@example.com,North,2023-01-15,True
1,C002,Bob Smith,bob@test.com,South,2023-02-20,True
2,C003,Charlie Brown,NaN,East,2023-03-10,False
3,C004,David Wilson,david@example.com,West,2023-04-05,True
4,C005,Eva Garcia,NaN,Central,2023-05-12,False


## 1.5 Fill Missing Regions
Fill missing region with the string 'Unknown'.

In [18]:
cust_df['region'] = cust_df['region'].replace(['None', 'nan', ''], 'Unknown')
print("Value counts for region:")
print(cust_df['region'].value_counts())
cust_df.head()

Value counts for region:
region
North      2
South      2
East       2
West       1
Central    1
Name: count, dtype: int64


,customer_id,name,email,region,signup_date,is_valid_email
0,C001,Alice Johnson,alice@example.com,North,2023-01-15,True
1,C002,Bob Smith,bob@test.com,South,2023-02-20,True
2,C003,Charlie Brown,NaN,East,2023-03-10,False
3,C004,David Wilson,david@example.com,West,2023-04-05,True
4,C005,Eva Garcia,NaN,Central,2023-05-12,False


## 2. Save Cleaned Customers

In [19]:
cust_df.to_csv(DATA_DIR / 'customers_clean.csv', index=False)
print("Successfully saved customers_clean.csv")

Successfully saved customers_clean.csv


## 3. Load Orders Data

In [20]:
orders_df = pd.read_csv(DATA_DIR / 'orders.csv')
print(f"Initial rows: {len(orders_df)}")
orders_df.head()

Initial rows: 25


,order_id,customer_id,product,amount,order_date,status
0,OR1001,C003,P005,535.07,2023-09-10,Delivered
1,OR1002,C005,P004,NaN,28/09/2023,Delivered
2,OR1003,C003,P001,NaN,01/04/2023,Cancelled
3,OR1004,C004,P004,1152.35,2023-07-20,Shipped
4,OR1005,C004,P005,944.46,19/09/2023,Pending


## 3.1 Parse Order Dates
Parse `order_date` supporting multiple formats (YYYY-MM-DD, DD/MM/YYYY, MM-DD-YYYY) using a custom parser.

In [21]:
def custom_date_parser(date_str):
    if pd.isna(date_str) or date_str == '':
        return pd.NaT
    for fmt in ['%Y-%m-%d', '%d/%m/%Y', '%m-%d-%Y', '%b %d, %Y']:
        try:
            return pd.to_datetime(date_str, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.to_datetime(date_str, errors='coerce')

orders_df['order_date'] = orders_df['order_date'].apply(custom_date_parser)
print(f"Null dates after parsing: {orders_df['order_date'].isna().sum()}")
orders_df.head()

Null dates after parsing: 0


,order_id,customer_id,product,amount,order_date,status
0,OR1001,C003,P005,535.07,2023-09-10,Delivered
1,OR1002,C005,P004,NaN,2023-09-28,Delivered
2,OR1003,C003,P001,NaN,2023-04-01,Cancelled
3,OR1004,C004,P004,1152.35,2023-07-20,Shipped
4,OR1005,C004,P005,944.46,2023-09-19,Pending


## 3.2 Drop Unrecoverable Rows
Drop rows where both `customer_id` and `order_id` are null.

In [22]:
before_count = len(orders_df)
orders_df = orders_df.dropna(subset=['customer_id', 'order_id'], how='all')
print(f"Unrecoverable rows dropped: {before_count - len(orders_df)}")
orders_df.head()

Unrecoverable rows dropped: 0


,order_id,customer_id,product,amount,order_date,status
0,OR1001,C003,P005,535.07,2023-09-10,Delivered
1,OR1002,C005,P004,NaN,2023-09-28,Delivered
2,OR1003,C003,P001,NaN,2023-04-01,Cancelled
3,OR1004,C004,P004,1152.35,2023-07-20,Shipped
4,OR1005,C004,P005,944.46,2023-09-19,Pending


## 3.3 Impute Missing Amounts
Fill missing `amount` with the median amount grouped by `product`.

In [23]:
orders_df['amount'] = pd.to_numeric(orders_df['amount'], errors='coerce')
orders_df['amount'] = orders_df.groupby('product')['amount'].transform(lambda x: x.fillna(x.median()))
print(f"Null amounts remaining: {orders_df['amount'].isna().sum()}")
orders_df.head()

Null amounts remaining: 0


,order_id,customer_id,product,amount,order_date,status
0,OR1001,C003,P005,535.07,2023-09-10,Delivered
1,OR1002,C005,P004,915.49,2023-09-28,Delivered
2,OR1003,C003,P001,1119.67,2023-04-01,Cancelled
3,OR1004,C004,P004,1152.35,2023-07-20,Shipped
4,OR1005,C004,P005,944.46,2023-09-19,Pending


## 3.4 Normalize Status
Normalize status to: {'completed', 'pending', 'cancelled', 'refunded'}.

In [24]:
status_mapping = {
    'shipped': 'completed',
    'delivered': 'completed',
    'done': 'completed',
    'pending': 'pending',
    'cancelled': 'cancelled',
    'canceled': 'cancelled',
    'refunded': 'refunded'
}
orders_df['status'] = orders_df['status'].str.lower().str.strip().map(status_mapping).fillna(orders_df['status'].str.lower())
print("Value counts for status:")
print(orders_df['status'].value_counts())
orders_df.head()

Value counts for status:
status
completed    10
cancelled     9
pending       6
Name: count, dtype: int64


,order_id,customer_id,product,amount,order_date,status
0,OR1001,C003,P005,535.07,2023-09-10,completed
1,OR1002,C005,P004,915.49,2023-09-28,completed
2,OR1003,C003,P001,1119.67,2023-04-01,cancelled
3,OR1004,C004,P004,1152.35,2023-07-20,completed
4,OR1005,C004,P005,944.46,2023-09-19,pending


## 3.5 Derive Year-Month
Add derived column `order_year_month` (YYYY-MM).

In [25]:
orders_df['order_year_month'] = orders_df['order_date'].dt.strftime('%Y-%m')
orders_df.head()

,order_id,customer_id,product,amount,order_date,status,order_year_month
0,OR1001,C003,P005,535.07,2023-09-10,completed,2023-09
1,OR1002,C005,P004,915.49,2023-09-28,completed,2023-09
2,OR1003,C003,P001,1119.67,2023-04-01,cancelled,2023-04
3,OR1004,C004,P004,1152.35,2023-07-20,completed,2023-07
4,OR1005,C004,P005,944.46,2023-09-19,pending,2023-09


## 4. Save Cleaned Orders

In [26]:
orders_df.to_csv(DATA_DIR / 'orders_clean.csv', index=False)
print("Successfully saved orders_clean.csv")

Successfully saved orders_clean.csv
